# Notebook 2: Chains – Linking LLM Steps Like LEGO

**🎯 Goal:** Learn how to connect multiple LLM calls and other components together into a single, coherent application using Chains.

## 🧩 Concept Overview

Chains are the heart of LangChain. They allow you to combine multiple steps in a logical sequence. Think of it like an assembly line:

- **Step 1:** An LLM generates a product name.
- **Step 2:** The product name is passed to another LLM to write a description for it.
- **Step 3:** The description is passed to a final LLM to generate marketing hashtags.

Chains automate this entire workflow. The two main types we'll cover are:
- **`LLMChain`**: The fundamental building block (Prompt + Model).
- **`SequentialChain`**: A chain that runs other chains in a specific order.

## 🖼️ Visual Diagram

Here’s a `SimpleSequentialChain` that writes a blog post:

```
             +-----------------+      +----------------------+
Topic Input => | Chain 1:        | => | Chain 2:             | => Final Summary
             | Generate Title  |      | Generate Summary     |
             | (LLM)           |      | from Title (LLM)     |
             +-----------------+      +----------------------+
```

## ⚙️ Setup

As before, we'll load our environment variables and initialize our LLM.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain, SimpleSequentialChain, SequentialChain

load_dotenv()

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.5)

## 1. `LLMChain`: The Basic Building Block

An `LLMChain` is the simplest and most common type of chain. It takes a prompt template, formats it with user input, and returns the response from an LLM. We used a simplified version of this in Notebook 1 with the `|` operator. Here's the formal way to define it.

In [ ]:
# 📝 Prompt to generate a title for a blog post
title_prompt = PromptTemplate.from_template(
    "Generate a catchy, clickbait-style title for a blog post about {topic}."
)

# 🔗 Create the LLMChain for generating a title
title_chain = LLMChain(llm=llm, prompt=title_prompt)

# 🚀 Run the chain
response = title_chain.invoke({"topic": "the future of AI"})
print(response)

## 2. `SimpleSequentialChain`: Your First Multi-Step Chain

This chain is the easiest way to link two or more chains together. It takes the output from the first chain and uses it as the input for the second chain. It's simple because each step has only **one input** and **one output**.

In [ ]:
# 📝 Prompt to generate a summary based on a title
summary_prompt = PromptTemplate.from_template(
    "Write a 2-sentence summary for a blog post with the title: '{title}'."
)

# 🔗 Create the LLMChain for generating a summary
# Note: The input variable here ('title') must match the output of the previous chain.
summary_chain = LLMChain(llm=llm, prompt=summary_prompt)

# 엮다 (엮다 - yeokkda - to weave/link) the two chains together
sequential_chain = SimpleSequentialChain(
    chains=[title_chain, summary_chain],
    verbose=True  # Set to True to see the inner workings of the chain!
)

# 🚀 Run the full chain
final_response = sequential_chain.invoke({"topic": "why cats are secretly plotting world domination"})
print(final_response)

## 🔧 Hands-On Exercise

**Goal:** Create a 3-step `SimpleSequentialChain` that does the following:
1.  **Step 1:** Takes a `company_name` and generates a product idea.
2.  **Step 2:** Takes the `product_idea` and writes a 20-word marketing slogan for it.
3.  **Step 3:** Takes the `slogan` and lists 3 potential target customer groups.

In [ ]:
# --- Your Code Here --- 

# 1. Product Idea Chain
product_prompt = PromptTemplate.from_template("Come up with a new, innovative product idea for the company '{company_name}'.")
product_chain = LLMChain(llm=llm, prompt=product_prompt, output_key="product_idea") # output_key is optional for SimpleSequentialChain

# 2. Slogan Chain
slogan_prompt = PromptTemplate.from_template("Create a 20-word marketing slogan for this product: {product_idea}")
slogan_chain = LLMChain(llm=llm, prompt=slogan_prompt, output_key="slogan")

# 3. Target Audience Chain
audience_prompt = PromptTemplate.from_template("List three potential target customer groups for a product with the slogan: '{slogan}'.")
audience_chain = LLMChain(llm=llm, prompt=audience_prompt, output_key="audience")

# 엮다 - Link them all together
exercise_chain = SimpleSequentialChain(
    chains=[product_chain, slogan_chain, audience_chain],
    verbose=True
)

# 🚀 Run it!
exercise_result = exercise_chain.invoke({"company_name": "Nike"})
print(exercise_result)

## 🐞 Bug Bounty

The `SequentialChain` below is broken. It's more powerful than `SimpleSequentialChain` because it can handle multiple inputs and outputs, but it requires you to be explicit about the variable names.

**Hint:** Check the `input_variables` and `output_variables` for each chain and make sure they match up correctly.

In [ ]:
# This chain takes a topic and a writing style, generates a title, then a summary.
# It needs to remember the original 'style' for the second step.

# --- BROKEN CODE ---
title_prompt_broken = PromptTemplate.from_template("Generate a title for a blog post about {topic} in a {style} style.")
title_chain_broken = LLMChain(llm=llm, prompt=title_prompt_broken, output_key="blog_title")

summary_prompt_broken = PromptTemplate.from_template("Summarize this blog post titled '{blog_title}' in a {style} tone.")
# The bug is here! This chain doesn't know where to get 'style' from.
summary_chain_broken = LLMChain(llm=llm, prompt=summary_prompt_broken, output_key="summary")

# broken_sequential_chain = SequentialChain(
#     chains=[title_chain_broken, summary_chain_broken],
#     input_variables=["topic", "style"],
#     output_variables=["blog_title", "summary"],
#     verbose=True
# )
# try:
#     broken_sequential_chain.invoke({"topic": "meditation", "style": "sarcastic"})
# except Exception as e:
#     print(f"It failed! Error: {e}")

# --- FIXED CODE ---
# The fix is to use a chain type that can handle this, or structure the prompts differently.
# The SequentialChain is designed for linear data flow. A more complex graph structure is better here,
# but we can simulate it by passing variables through.

from langchain.chains import SequentialChain

# The input to this chain will be 'topic' and 'style'
# The output of this chain will be 'blog_title'
title_chain_fixed = LLMChain(
    llm=llm, 
    prompt=PromptTemplate.from_template("Generate a title for a blog post about {topic} in a {style} style."), 
    output_key="blog_title"
)

# The input to this chain will be 'blog_title' and 'style'
# The output will be 'summary'
summary_chain_fixed = LLMChain(
    llm=llm, 
    prompt=PromptTemplate.from_template("Summarize this blog post titled '{blog_title}' in a {style} tone."),
    output_key="summary"
)

# The SequentialChain will correctly map the inputs and outputs
fixed_sequential_chain = SequentialChain(
    chains=[title_chain_fixed, summary_chain_fixed],
    input_variables=["topic", "style"], # All inputs for the entire sequence
    output_variables=["blog_title", "summary"], # All outputs from the entire sequence
    verbose=True
)

result = fixed_sequential_chain.invoke({"topic": "meditation", "style": "witty and cynical"})
print("\n--- Fixed Code Output ---")
print(result)

## 💡 Pro Tip

Always use `verbose=True` when debugging chains! It prints out the inputs and outputs of each step, making it much easier to see where something went wrong. You can see the flow of variables and the exact text being passed to the LLM at each stage.

## 🏁 Summary

You've now mastered the art of linking LLM calls together!

1.  **Chains are sequences:** They allow you to build complex logic by making the output of one step the input of another.
2.  **Start with `SimpleSequentialChain`:** It's great for linear, single-input/single-output workflows.
3.  **Use `SequentialChain` for more control:** When you need to manage multiple inputs and outputs across steps, `SequentialChain` gives you the necessary control by explicitly defining `input_variables` and `output_variables`.